# Laboratorio 1 - Exploración, preparación y regresión lineal

## AlpesPlanck

**Integrantes:**
- Daniel Esteban Pardo Pardo
- Samuel Andrés Molina Luna

**Curso:** ISIS2611 - Aprendizaje de Máquina
**Fecha:** 31 de Agosto 2026

# 1. Contexto
El Instituto AlpesPlanck de Biogeoquímica registra variables meteorológicas cada 10 minutos en su estación de Jena, Alemania, desde 2003. A partir de un histórico de datos que se nos ha proporcionado, realizaremos una exploración de los mismos para entender su composición y validar los principios de calidad (unicidad, completitud, validez y consistencia) estudiados en el curso. <br><br>
De esta manera podremos llevar a cabo un proceso de ingeniería de datos que nos permitirá preparar los datos proporcionados para, finalmente, elaborar dos modelos de **regresión lineal** con el uso de **pipelines**. Estos modelos serán comparados a partir de métricas estadísticas precisas para elegir el parezca tener mejor desempeño y aplicarlo sobre un conjunto de datos sin etiquetas.      

# 2. Carga de datos

### Librerías a usar en el transcurso del laboratorio
Librerías de Python para el procesamiento y analisis de datos como:<br>

- Pandas
- Scikit-Learn
- Matplotlib, Seaborn


In [2]:
# Importación de las librerías a usar
import pandas as pd
import matplotlib as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Matplotlib is building the font cache; this may take a moment.


### Cargar datos a DataFrames

##### DataSet de entrenamiento

In [3]:
data = pd.read_csv('data/Datos Lab 1.csv')
data.head()

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,...,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,...,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,...,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,...,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,...,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


#### DataSet de prueba (Sin etiquetas)

In [4]:
data_test = pd.read_csv('data/Datos Test Lab 1.csv')
data_test.head()

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento
0,01.01.2016,998.9860,995.57,1000.60,1.1082,96.1493,90.8,98.7,2.0034,0.9649,...,0.8415,-0.8328,0.0927,173.6501,144,2016,1,invierno,January,S
1,02.01.2016,991.4241,989.27,995.51,1.9413,92.3236,85.8,96.8,2.6562,1.9886,...,1.3604,1.1161,0.6688,30.9301,144,2016,2,invierno,January,NE
2,03.01.2016,984.0232,974.12,989.39,4.5841,83.6056,73.9,94.7,6.9967,2.4992,...,1.2558,-0.7923,1.2638,122.0847,144,2016,3,invierno,January,SE
3,04.01.2016,968.2449,966.52,974.02,1.9034,85.4688,75.5,93.8,5.8148,2.0932,...,1.4544,-1.6217,0.7020,156.5944,144,2016,4,invierno,January,SE
4,05.01.2016,970.0099,967.85,972.90,1.6668,94.3521,92.0,96.8,1.4253,1.8380,...,0.7284,1.6549,0.7109,23.2458,144,2016,5,invierno,January,NE


Notamos que el DataFrame que corresponde a los datos de entrenamiento tiene una columna más que el de los datos de prueba. Esta columna adicional corresponde a `temp_max_manana`; nuestra variable objetivo y la que completaremos en el conjunto de datos de prueba una vez tengamos nuestro modelo de regresión lineal definido.

# 3. Exploración de datos

### 3.1 Dimensiones y tipos de datos

El conjunto de datos de entrenamiento tiene en total 2.576 registros y 27 columnas. Vemos que la
mayoria de las variables son numericas como se puede observar en las variables mediciones de presion, humedad, viento 
y rafagas, mientras que fecha, estacion_anio, mes y sector_viento son 
variables de tipo texto.

In [5]:
print(f"Dimensiones: {data.shape[0]} filas, {data.shape[1]} columnas")
data.info()

Dimensiones: 2576 filas, 27 columnas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              2504 non-null   object 
 1   presion_media      2501 non-null   float64
 2   presion_min        2502 non-null   float64
 3   presion_max        2512 non-null   float64
 4   presion_desv       2496 non-null   float64
 5   humedad_media      2502 non-null   float64
 6   humedad_min        2496 non-null   float64
 7   humedad_max        2490 non-null   float64
 8   humedad_desv       2515 non-null   float64
 9   viento_media       2490 non-null   float64
 10  viento_min         2485 non-null   float64
 11  viento_max         2495 non-null   float64
 12  viento_desv        2504 non-null   float64
 13  rafaga_media       2498 non-null   float64
 14  rafaga_min         2504 non-null   float64
 15  rafaga_max         2503 non-null   

### 3.2 Valores nulos

Ahora se revisa la cantidad de valores nulos por columna para poder sacar el porcentaje de datos faltantes y decidir como tratarlos en la etapa de preparacion.

In [6]:
nulos = data.isnull().sum()
porcentaje_nulos = (nulos / len(data) * 100).round(2)
pd.DataFrame({'nulos': nulos, 'porcentaje': porcentaje_nulos}).sort_values('nulos', ascending=False)

,nulos,porcentaje
temp_max_manana,95,3.69
viento_min,91,3.53
mes,88,3.42
estacion_anio,88,3.42
anio,86,3.34
humedad_max,86,3.34
viento_media,86,3.34
viento_max,81,3.14
presion_desv,80,3.11
humedad_min,80,3.11


Vemos que todas las columnas presentan valores nulos en los porcentajes entre 2.37 a 3.69 casi 4%, tambien se tiene la variable temp_max_manana que tiene el mayor numero de valores nulos con un total de 95 (3.69%).

### 3.3 Duplicados

Aqui se revisa si existen filas completamente duplicadas así como fechas repetidas ya que cada fila deberia representar la observacion de un unico día.

In [8]:
print("Filas duplicadas:", data.duplicated().sum())
print("Fechas duplicadas:", data['fecha'].duplicated().sum())
data[data['fecha'].duplicated(keep=False)].sort_values('fecha').head(10)

Filas duplicadas: 4
Fechas duplicadas: 89


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
21,2009-01-22,9784.115000,971.48,983.54,3.9853,86.731100,69.110,97.6,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770000
2563,2009-01-22,9784.115000,971.48,983.54,3.9853,86.731100,69.110,97.6,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770000
63,2009-03-05,959.860700,958.67,962.54,0.8964,94.294400,86.000,97.8,2.8296,1.29080,...,1.1131,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.370000
2574,2009-03-05,959.860700,958.67,962.54,0.8964,NaN,86.000,97.8,2.8296,1.29080,...,1.1131,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.956009
121,2009-05-02,998.118900,995.99,999.41,0.9205,77.500000,51.090,99.6,18.9020,1.84440,...,NaN,0.4668,23.7431,144.0,NaN,122.0,primavera,october,NaN,20.890000
2568,2009-05-02,998.855231,995.99,999.41,0.9205,77.500000,51.090,99.6,18.9020,1.84440,...,1.0611,0.4668,23.7431,144.0,2009.0,122.0,primavera,october,NE,20.890000
2575,2009-12-08,983.849000,979.45,991.40,3.7836,1.021309,0.801,96.8,5.9641,4.53492,...,-0.6218,NaN,233.2095,144.0,2009.0,342.0,verano,Enero,SO,7.120000
341,2009-12-08,983.849000,979.45,991.40,3.7836,0.877951,0.801,96.8,5.9641,4.53492,...,-0.6218,-0.8314,233.2095,144.0,2009.0,342.0,verano,Enero,SO,7.120000
2566,2010-08-14,991.272748,989.22,991.29,0.4346,84.386400,65.120,95.7,9.4681,1.70860,...,0.7872,0.6511,39.5952,144.0,2010.0,226.0,verano,August,NE,19.500000
590,2010-08-14,990.482800,989.22,991.29,0.4346,84.386400,65.120,95.7,9.4681,1.70860,...,0.7872,0.6511,39.5952,144.0,2010.0,226.0,verano,August,NE,19.500000


Se identifican 4 filas y 89 fechas duplicadas, tambien se observa que la mayoria de estas fechas corresponden a registros con valores distintos en las demas columnas, osea no son copias exactas como tal sino errores de asignacion de fecha entre observaciones de dias distintos. Adicionalmente a esto, una de las 4 filas duplicadas contiene el valor de presion_media con 9784.115000 identificado previamente lo que implica que ese error esta presente en ambas copias.

Por ultimo se detecta que las columnas mes y estacion_anio son inconsistentes con dia_del_anio, por ejemplo se presenta el dia 22 del año que corresponde a enero e invierno, y aqui aparece etiquetado como octubre/verano, lo que indica que estas columnas no son confiables y deberan recalcularse a partir de fecha o dia_del_anio en la etapa de preparacion.